# Synthetic control: IRA & CHIPS Act and non-residential investment

Question: at the country level, does US investment in non-residential structures break away from its synthetic twin after August 2022?
We follow the method of Abadie, Diamond and Hainmueller (2010), with permutation inference. The computations are in [`src/synth.py`](../src/synth.py).

In [1]:
import sys

import polars as pl

sys.path.append('../src')
import data_processing as dp
import synth
from plot_functions import set_theme, plot_ranking, plot_synth, plot_gaps, plot_specs

set_theme()
data = dp.load_processed()
OECD = data['oecd_gfcf_index2019_quarterly']

## 1. Design

The synthetic twin is a weighted average of the 20 other OECD countries, $\hat{Y}_{USA,t} = \sum_j w_j\, Y_{j,t}$ with $w_j \geq 0$ and $\sum_j w_j = 1$. The weights are chosen to track the United States as closely as possible from 2015-Q1 to 2022-Q2.
The method assumes that, without the laws, the United States would have kept tracking this twin, that the donors are not affected by the laws and that no other US-specific shock occurs in 2022-Q3.
It complements the DiD in two ways: the unit is the country rather than the sector, and the outcome is a volume already adjusted for inflation by the OECD. This outcome, GFCF in non-residential buildings and structures (100 = 2019), however mixes offices, infrastructure and factories, so the expected effect is diluted.

## 2. Weights

In [2]:
w = synth.weights(OECD).filter(pl.col('weight') > 0.001)
plot_ranking(w, 'country', 'weight', n=w.height, title='Donor weights in the synthetic twin',
             subtitle='Non-zero weights, fitted on 2015-Q1 → 2022-Q2', xtitle='Weight')

The twin relies on 8 countries, dominated by Ireland (**0.27**), then Japan (0.15), Sweden, Canada and Finland (0.13 each).
Ireland's weight is a problem because its series is very volatile, and section 6 tests the result without it.

## 3. Pre-treatment fit

In [3]:
s = synth.synthetic(OECD)
plot_synth(s, title='United States and synthetic twin',
           subtitle="GFCF in non-residential buildings and structures, volume, 100 = 2019")

In [4]:
pl.DataFrame([synth.summary(s)]).with_columns(pl.selectors.numeric().round(2))

rmspe_pre,rmspe_post,ratio,mean_gap,peak_gap,peak_quarter
f64,f64,f64,f64,f64,str
2.44,9.73,3.99,8.31,14.23,"""2024-Q1"""


The pre-treatment RMSPE is **2.44** index points. The twin tracks the United States closely from 2015 to 2019, but diverges in 2020-2022, with the United States falling up to 6.6 points below it in 2021-Q4.
The fit is therefore moderate, and the positive gap observed after 2022 may partly reflect a rebound from this trough.

## 4. Estimation

In [5]:
plot_gaps(s, title='Gap between the United States and its synthetic twin', subtitle="Index points (100 = 2019)")

After 2022-Q3, the United States exceeds its twin by **+8.3 points** on average, with a peak of +14.2 points in 2024-Q1, then the gap falls back to +6.6 in 2026-Q2.
The post-treatment RMSPE (9.73) is 3.99 times the pre-treatment RMSPE.

## 5. Inference

In [6]:
gaps, ratios = synth.space_placebos(OECD)
plot_gaps(gaps, title='Gaps for the United States and in-space placebos',
          subtitle='Each country treated in turn, United States excluded from placebo donors')

In [7]:
ratios.head(6).with_columns(pl.selectors.float().round(2))

rank,country,rmspe_pre,rmspe_post,ratio
u32,str,f64,f64,f64
1,"""ITA""",4.08,53.53,13.13
2,"""FRA""",1.02,13.13,12.91
3,"""AUT""",1.79,17.81,9.96
4,"""ESP""",2.12,14.4,6.78
5,"""NLD""",1.39,8.36,6.03
6,"""USA""",2.44,9.73,3.99


Ranked by post / pre RMSPE ratio, the United States (3.99) comes only **6th out of 21**, behind Italy, France, Austria, Spain and the Netherlands.
Several European countries thus diverge more from their own twin after 2022.

In [8]:
synth.p_values(ratios).with_columns(pl.col('p').round(3))

placebos,countries,usa_rank,p
str,i64,i64,f64
"""All countries""",21,6,0.286
"""Pre RMSPE ≤ 2 × USA""",14,6,0.429


The rank p-value is **0.286** (6/21), against a minimum possible of 1/21 ≈ 0.048 with 21 countries.
Removing the poorly fitted placebos (pre RMSPE more than twice that of the United States) leaves 14 countries and p rises to 0.429. The US gap does not stand out from the placebos.

## 6. Robustness

In [9]:
plot_specs(synth.robustness(OECD, data['oecd_donor_fit']), 'specification', 'mean_gap', 'Main (2022-Q3)',
           pct=False, title='Mean gap by specification',
           subtitle="Gap between the United States and its twin after treatment, index points", xtitle="Index points")

Removing the other donors one at a time, or keeping only the 5 best-correlated, leaves the gap between +7.7 and +9.2 points. Without Ireland, it falls to **+1.7**, so the result rests on a single donor.
The in-time placebo (fake date in 2019-Q1) gives −2.2 points. Moving the treatment forward to 2021-Q3 reduces the gap to +4.1, as the post period then includes the 2021-2022 trough.

## 7. Limitations

| Threat | Risk | Effect on the estimate |
|---|---|---|
| European energy crisis | Nearly two thirds of the weight goes to European countries (Ireland, Sweden, Finland, Spain, Czechia), held back in 2022-2023. | Twin too low, effect overestimated. |
| Spillover | The IRA attracts projects that would otherwise have happened in Europe or Canada. | Donors lower, effect overestimated. |
| Outcome too broad | Offices and infrastructure weigh far more than the factories targeted by the laws. | Effect diluted, hence underestimated. |
| US-specific shocks | Higher Fed rates, rebound after the 2021 trough. | Ambiguous direction: rates slow investment, the rebound inflates the gap. |

## 8. Takeaways

- **Country effect**: +8.3 points on average after 2022-Q3 (peak of +14.2 in 2024-Q1), far below the sector-level DiD effect (+189% to +452%), which is consistent with a broad aggregate.
- **Fit**: good until 2019, moderate in 2020-2022, with the United States below its twin just before the laws; part of the gap may be a rebound.
- **Inference**: 6th out of 21, p = 0.286 (0.429 without the poorly fitted placebos); the US gap does not stand out from those of other countries.
- **Fragility**: without Ireland, the gap falls to +1.7 points.
- **Link with the DiD**: the synthetic control does not contradict the strong effect on the treated sector, but this effect is not clearly visible at the country level.